In [1]:
import sys
import os
import importlib
import json
from pathlib import Path

In [2]:
def run_test(test_module_name, original_ifc_path, ROOT_DIR, edited_ifc_path=None, model_output=None):
    """
    Run a test module and display results.

    Args:
        test_module_name: e.g., 'test_01'
        original_ifc_path: path to original IFC file (relative to repo root or absolute)
        edited_ifc_path: path to edited IFC file (relative to repo root or absolute)
        model_output: optional model output dict
    """
    if edited_ifc_path is None:
        edited_ifc_path = original_ifc_path

    original_path = Path(original_ifc_path)
    edited_input_path = Path(edited_ifc_path)

    if not original_path.is_absolute():
        original_path = ROOT_DIR / original_path
    if not edited_input_path.is_absolute():
        edited_input_path = ROOT_DIR / edited_input_path

    ifc_path = os.path.normpath(str(original_path))
    edited_path = os.path.normpath(str(edited_input_path))

    print(f"\n{'='*60}")
    print(f"Running: {test_module_name}")
    # print(f"Original: {ifc_path}")
    # print(f"Edited:   {edited_path}")
    print(f"{'='*60}")

    if not os.path.exists(ifc_path):
        print(f"ERROR: Original IFC file not found: {ifc_path}")
        return None
    if not os.path.exists(edited_path):
        print(f"ERROR: Edited IFC file not found: {edited_path}")
        return None

    root_str = str(ROOT_DIR)
    if root_str not in sys.path:
        sys.path.insert(0, root_str)

    try:
        module_candidates = [test_module_name, f"data.tests.{test_module_name}"]
        mod = None
        for candidate in module_candidates:
            try:
                mod = importlib.import_module(candidate)
                importlib.reload(mod)  
                break
            except ModuleNotFoundError as e:
                if e.name != candidate:
                    raise
        if mod is None:
            raise ModuleNotFoundError(
                f"Could not import '{test_module_name}'. Tried: {', '.join(module_candidates)}"
            )

        metrics = mod.execute_test(ifc_path, edited_path, model_output)

        print(f"\nMetrics:")
        for k, v in metrics.items():
            status = '  PASS' if (isinstance(v, bool) and v) or (isinstance(v, float) and v > 0) else '  FAIL'
            print(f"  {k:25s} = {v!s:10s} {status}")

        score = sum(v if isinstance(v, float) else float(v) for v in metrics.values()) / len(metrics)
        print(f"\n  Overall score: {score:.4f}")
        return metrics
    except Exception as e:
        print(f"ERROR: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        return None

In [3]:
# Direct Tasks
ROOT_DIR = Path.cwd().resolve().parents[2]

IFC_DIR = Path("data") / "ifc"
EDITED_DIR = Path("runs") / "gemini-3-flash" / "edited_ifc_gemini-3-flash-preview"

# Sanity check: original == edited should give all zeros
print("SANITY CHECK: original == edited (expect all FAIL / score 0.0)")
ifc_original_sanity = str(IFC_DIR / "basic_tasks.ifc")
for test_name in ["test_01", "test_02", "test_03", "test_04", "test_05"]:
    run_test(test_name, ifc_original_sanity, ROOT_DIR, edited_ifc_path=ifc_original_sanity)

SANITY CHECK: original == edited (expect all FAIL / score 0.0)

Running: test_01

Metrics:
  object_exists             = False        FAIL
  right_location            = False        FAIL
  right_dimensions          = False        FAIL
  integrity_constraint      = 0.0          FAIL

  Overall score: 0.0000

Running: test_02

Metrics:
  object_exists             = False        FAIL
  right_location            = False        FAIL
  right_dimensions          = False        FAIL
  integrity_constraint      = 0.0          FAIL

  Overall score: 0.0000

Running: test_03

Metrics:
  object_exists             = False        FAIL
  right_location            = False        FAIL
  right_dimensions          = False        FAIL
  integrity_constraint      = 0.0          FAIL

  Overall score: 0.0000

Running: test_04

Metrics:
  object_exists             = False        FAIL
  right_location            = False        FAIL
  right_dimensions          = False        FAIL
  integrity_constraint      = 

In [13]:
id = 1
id1 = id - 1
test_module_name = f"test_0{id}"
ifc_original = str(IFC_DIR / "basic_tasks.ifc")
edited = str(EDITED_DIR / str(id1) / "basic_tasks_0.ifc")
metrics_01 = run_test(test_module_name, ifc_original, ROOT_DIR, edited_ifc_path=edited)


Running: test_01

Metrics:
  object_exists             = True         PASS
  right_location            = False        FAIL
  right_dimensions          = False        FAIL
  integrity_constraint      = 0.0          FAIL

  Overall score: 0.2500


In [ ]:
id = 2
id1 = id - 1
test_module_name = f"test_0{id}"
ifc_original = str(IFC_DIR / "basic_tasks.ifc")
edited = str(EDITED_DIR / str(id1) / "basic_tasks_0.ifc")
metrics_01 = run_test(test_module_name, ifc_original, ROOT_DIR, edited_ifc_path=edited)

In [14]:
# Topological Tasks
ROOT_DIR = Path.cwd().resolve().parents[2]

IFC_DIR = Path("data") / "ifc"
EDITED_DIR = Path("runs") / "gemini-3-flash" / "edited_ifc_gemini-3-flash-preview"

# Sanity check: original == edited should give all zeros
print("SANITY CHECK: original == edited (expect all FAIL / score 0.0)")
for id in range(1, 6):
    test_module_name = f"topological_test_{id:02d}"
    ifc_original_sanity = str(IFC_DIR / "01" / "02" / f"01_02_{id:03d}.ifc")
    run_test(test_module_name, ifc_original_sanity, ROOT_DIR, edited_ifc_path=ifc_original_sanity)

SANITY CHECK: original == edited (expect all FAIL / score 0.0)

Running: topological_test_01

Metrics:
  object_exists             = False        FAIL
  right_location            = False        FAIL
  right_dimensions          = False        FAIL
  integrity_constraint      = 0.0          FAIL

  Overall score: 0.0000

Running: topological_test_02

Metrics:
  object_exists             = False        FAIL
  right_location            = False        FAIL
  right_dimensions          = False        FAIL
  integrity_constraint      = 0.0          FAIL

  Overall score: 0.0000

Running: topological_test_03

Metrics:
  object_exists             = False        FAIL
  right_location            = False        FAIL
  right_dimensions          = False        FAIL
  integrity_constraint      = 0.0          FAIL

  Overall score: 0.0000

Running: topological_test_04

Metrics:
  object_exists             = False        FAIL
  right_location            = False        FAIL
  right_dimensions          = 

In [15]:
id = 4
run_index = 40 + id - 1
test_module_name = f"topological_test_{id:02d}"
ifc_original = str(IFC_DIR / "01" / "02" / f"01_02_{id:03d}.ifc")
edited = str(EDITED_DIR / str(run_index) / f"01_02_{id:03d}_0.ifc")
metrics_01 = run_test(test_module_name, ifc_original, ROOT_DIR, edited_ifc_path=edited)


Running: topological_test_04

Metrics:
  object_exists             = True         PASS
  right_location            = True         PASS
  right_dimensions          = True         PASS
  integrity_constraint      = 0.6667       PASS

  Overall score: 0.9167


In [52]:
# Geometric Tasks
ROOT_DIR = Path.cwd().resolve().parents[2]

IFC_DIR = Path("data") / "ifc"
EDITED_DIR = Path("runs") / "gemini-3-flash" / "edited_ifc_gemini-3-flash-preview"

# Sanity check: original == edited should give all zeros
print("SANITY CHECK: original == edited (expect all FAIL / score 0.0)")
for id in range(1, 6):
    test_module_name = f"geometry_test_{id:02d}"
    if id == 5:
        ifc_original_sanity = str(IFC_DIR / "empty.ifc")
    else:
        ifc_original_sanity = str(IFC_DIR / "01" / "01" / f"01_01_{id:03d}.ifc")
    run_test(test_module_name, ifc_original_sanity, ROOT_DIR, edited_ifc_path=ifc_original_sanity)

SANITY CHECK: original == edited (expect all FAIL / score 0.0)

Running: geometry_test_01

Metrics:
  object_exists             = False        FAIL
  right_location            = False        FAIL
  right_dimensions          = False        FAIL
  integrity_constraint      = 0.0          FAIL

  Overall score: 0.0000

Running: geometry_test_02

Metrics:
  object_exists             = False        FAIL
  right_location            = False        FAIL
  right_dimensions          = False        FAIL
  integrity_constraint      = 0.0          FAIL

  Overall score: 0.0000

Running: geometry_test_03

Metrics:
  object_exists             = False        FAIL
  right_location            = False        FAIL
  right_dimensions          = False        FAIL
  integrity_constraint      = 0.0          FAIL

  Overall score: 0.0000

Running: geometry_test_04

Metrics:
  object_exists             = False        FAIL
  right_location            = False        FAIL
  right_dimensions          = False       

In [20]:
id = 5
run_index = 20 + id - 1
test_module_name = f"geometry_test_{id:02d}"
if id == 5:
    ifc_original = str(IFC_DIR / "empty.ifc")
    edited = str(EDITED_DIR / str(run_index) / "empty_0.ifc")
else:
    ifc_original = str(IFC_DIR / "01" / "01" / f"01_01_{id:03d}.ifc")
    edited = str(EDITED_DIR / str(run_index) / f"01_01_{id:03d}_0.ifc")
metrics_01 = run_test(test_module_name, ifc_original, ROOT_DIR, edited_ifc_path=edited)


Running: geometry_test_05

Metrics:
  object_exists             = True         PASS
  right_location            = True         PASS
  right_dimensions          = True         PASS
  integrity_constraint      = 1.0          PASS

  Overall score: 1.0000
